In [70]:
import pandas as pd


default_file_path = "../../data/raw/"
save_file_path = "../../data/iterim/"


def convert_integer_column(df_column: pd.Series):
    converted_column = df_column.astype("string").str.replace(" ", "", regex=False).str.replace(",", ".", regex=False).str.replace("\u00a0", "", regex=False).str.replace("—", "-", regex=False)
    converted_column = pd.to_numeric(converted_column, errors="coerce")
    return converted_column.round(2)


def convert_datetime_column(df_column: pd.Series):
    converted_column = pd.to_datetime(df_column, errors="coerce", format="mixed")
    return converted_column


def convert_string_column(df_column: pd.Series):
    convert_column = df_column.astype("string").str.lower().str.strip().str.replace(" ", "_")
    return convert_column

In [71]:
recharge_df = pd.read_csv(f"{default_file_path}recharge.csv")

In [84]:
display(recharge_df)
display(recharge_df.dtypes)
display(recharge_df.describe())

,zone,recharge,t,record_id,revision,unit
0,water_intake_zone,7.47,2027-12-01,R-00254,1,mm/month
1,central_zone,23.87,2021-05-01,R-00328,1,mm/month
2,industrial_zone,6.16,2026-07-01,R-00090,1,mm/month
3,central_zone,7.05,2020-01-01,R-00243,1,mm/month
4,water_intake_zone,22.24,2026-10-01,R-00354,1,mm/month
...,...,...,...,...,...,...
377,industrial_zone,9.54,2020-06-01,R-00233,1,mm/month
378,industrial_zone,9.58,2026-06-01,R-00024,1,mm/month
379,central_zone,9.4,2020-08-01,R-00139,1,mm/month
380,central_zone,6.64,2028-08-01,R-00025,1,mm/month


zone                 string
recharge            Float64
t            datetime64[us]
record_id               str
revision              int64
unit                    str
dtype: object

,recharge,t,revision
count,364.0,364,364.000000
mean,11.169808,2023-11-09 18:47:28.351648,1.024725
min,0.0,2019-01-01 00:00:00,1.000000
25%,5.755,2021-05-01 00:00:00,1.000000
50%,9.345,2023-12-16 12:00:00,1.000000
75%,15.395,2026-06-01 00:00:00,1.000000
max,37.74,2028-12-01 00:00:00,2.000000
std,7.233268,NaN,0.155500


In [73]:
recharge_df["recharge"] = convert_integer_column(recharge_df["recharge"])
recharge_df["zone"] = convert_string_column(recharge_df["zone"])
recharge_df["t"] = convert_datetime_column(recharge_df["t"])

In [74]:
recharge_df["recharge"] = recharge_df.loc[recharge_df["recharge"].round(0) != -999, "recharge"]

In [75]:
recharge_df[recharge_df["recharge"].round(0) == -999]

,zone,recharge,t,record_id,revision,unit


In [76]:
recharge_df

,zone,recharge,t,record_id,revision,unit
0,water_intake_zone,7.47,2027-12-01,R-00254,1,mm/month
1,central_zone,23.87,2021-05-01,R-00328,1,mm/month
2,industrial_zone,6.16,2026-07-01,R-00090,1,mm/month
3,central_zone,7.05,2020-01-01,R-00243,1,mm/month
4,water_intake_zone,22.24,2026-10-01,R-00354,1,mm/month
...,...,...,...,...,...,...
377,industrial_zone,9.54,2020-06-01,R-00233,1,mm/month
378,industrial_zone,9.58,2026-06-01,R-00024,1,mm/month
379,central_zone,9.4,2020-08-01,R-00139,1,mm/month
380,central_zone,6.64,2028-08-01,R-00025,1,mm/month


In [77]:
recharge_df.loc[recharge_df["unit"] != "mm/month", ["recharge"]] = recharge_df["recharge"] * 100
recharge_df.loc[recharge_df["unit"] != "mm/month", ["unit"]] = "mm/month"

In [78]:
recharge_df[recharge_df["unit"] != "mm/month"]

,zone,recharge,t,record_id,revision,unit


In [79]:
recharge_df[~recharge_df["zone"].isin(["water_inteke_zone", "central_zone", "industrial_zone", "water_intake_zone"])]

,zone,recharge,t,record_id,revision,unit


In [82]:
recharge_df = recharge_df.dropna(subset=["recharge"])

In [83]:
recharge_df = recharge_df.drop_duplicates(subset=["recharge", "zone", "t"])

In [85]:
recharge_df.to_parquet(f"{save_file_path}recharge.parquet")